In [0]:
from pyspark.sql.functions import col, upper, trim, regexp_replace, translate, coalesce, when, lit

# 1. Read raw data from the Bronze Managed Table
table_name_bronze_sinesp = "bronze_sinesp_crime_data"
df_bronze_sinesp = spark.read.table(table_name_bronze_sinesp)

# 2. Helper function to clean numeric strings coming from Pandas
# Pandas converts nulls to the string "nan". This function restores them to genuine Nulls and casts to Integer.
def clean_numeric_col(column_name):
    return when(col(column_name) == "nan", lit(None)) \
           .otherwise(col(column_name).cast("double")).cast("integer")

# 3. Select relevant columns and unify the metrics
df_selected = df_bronze_sinesp.select(
    col("uf").alias("state_code"),
    col("municipio").alias("municipality_name_original"),
    col("evento").alias("crime_type"),
    col("data_referencia").alias("reference_date"),
    # coalesce picks the first non-null value between 'total' and 'total_vitima'
    coalesce(
        clean_numeric_col("total"), 
        clean_numeric_col("total_vitima"), 
        lit(0)
    ).alias("occurrences_count"),
    col("ingestion_timestamp"),
    col("source_system")
)

# 4. Data Cleansing and Standardization for the JOIN
accents = "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇáàâãäéèêëíìîïóòôõöúùûüç"
no_accents = "AAAAAEEEEIIIIOOOOOUUUUCAAAAAEEEEIIIIOOOOOUUUUC"

# 4.1 Remove accents, convert to uppercase, and trim spaces
df_silver_sinesp = df_selected.withColumn(
    "municipality_name_clean", 
    upper(trim(translate(col("municipality_name_original"), accents, no_accents)))
)

# 4.2 Remove special characters to match the IBGE pattern perfectly
df_silver_sinesp = df_silver_sinesp.withColumn(
    "municipality_name_clean",
    regexp_replace(col("municipality_name_clean"), "[^A-Z0-9 ]", "")
)

# 5. Save to the Silver Layer as a Managed Delta Table
table_name_silver_sinesp = "silver_sinesp_crime_data_cleansed"

df_silver_sinesp.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name_silver_sinesp)

print(f"Silver SINESP processing complete! {df_silver_sinesp.count()} records cleaned.")
display(df_silver_sinesp)